This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [1]:
import great_expectations as gx
import logging

In [2]:
import os
gx_context_root_dir=os.environ['GX_CONTEXT_ROOT_DIR']

In [3]:
context = gx.get_context(context_root_dir=gx_context_root_dir)
context.list_expectation_suites()

[ExpectationSuiteIdentifier::gfw-google-827.constraints.segs_activity.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.segs_activity.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.encounters.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.features_.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.features_.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.ssvids_identities.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.ssvids_identities.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.fishing_score_.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.fishing_score_.3-0-0,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.segs_activity.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.segs_activity.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.messages.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.alerts.messages.2-5,
 ExpectationSuiteIdentifier::gfw-google-827.constraints.seg

In [4]:
import yaml

In [5]:
from datetime import date,datetime

In [6]:
logging.basicConfig(level=logging.DEBUG, force = True)

In [7]:
with open(f"{gx_context_root_dir}/datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [8]:
datasource_config.get("project")

'gfw-google-827'

In [9]:
gx_project = datasource_config.get("project")
gx_datasource = context.get_datasource(gx_project)

In [10]:
gx_datasource.get_asset_names()

{'encounters-2.5',
 'encounters-3.0.0',
 'features_-2.5',
 'features_-3.0.0',
 'fishing_score_-2.5',
 'fishing_score_-3.0.0',
 'fragments-3.0.0',
 'loitering-2.5',
 'loitering-3.0.0',
 'messages-2.5',
 'messages-3.0.0',
 'messages_positions-2.5',
 'messages_positions-3.0.0',
 'messages_scored_-2.5',
 'messages_scored_-3.0.0',
 'messages_segmented_-2.5',
 'messages_segmented_-3.0.0',
 'satellite_timing_offsets-2.5',
 'satellite_timing_offsets-3.0.0',
 'segment_identity_daily_-2.5',
 'segment_identity_daily_-3.0.0',
 'segment_info-2.5',
 'segment_info-3.0.0',
 'segment_vessel-2.5',
 'segment_vessel-3.0.0',
 'segment_vessel_daily_-2.5',
 'segment_vessel_daily_-3.0.0',
 'segments-2.5',
 'segments-3.0.0',
 'segs_activity-2.5',
 'segs_activity-3.0.0',
 'segs_activity_daily-2.5',
 'segs_activity_daily-3.0.0',
 'ssvids_identities-2.5',
 'ssvids_identities-3.0.0',
 'ssvids_identities_daily-2.5',
 'ssvids_identities_daily-3.0.0',
 'stats_daily-2.5',
 'stats_daily-3.0.0',
 'vessel_info-2.5',
 've

In [11]:
context.list_expectation_suite_names()

['gfw-google-827.alerts.encounters.2-5',
 'gfw-google-827.alerts.encounters.3-0-0',
 'gfw-google-827.alerts.features_.2-5',
 'gfw-google-827.alerts.features_.3-0-0',
 'gfw-google-827.alerts.fishing_score_.2-5',
 'gfw-google-827.alerts.fishing_score_.3-0-0',
 'gfw-google-827.alerts.fragments.3-0-0',
 'gfw-google-827.alerts.loitering.2-5',
 'gfw-google-827.alerts.loitering.3-0-0',
 'gfw-google-827.alerts.messages.2-5',
 'gfw-google-827.alerts.messages.3-0-0',
 'gfw-google-827.alerts.messages_positions.2-5',
 'gfw-google-827.alerts.messages_positions.3-0-0',
 'gfw-google-827.alerts.messages_scored_.2-5',
 'gfw-google-827.alerts.messages_scored_.3-0-0',
 'gfw-google-827.alerts.messages_segmented_.2-5',
 'gfw-google-827.alerts.messages_segmented_.3-0-0',
 'gfw-google-827.alerts.satellite_timing_offsets.2-5',
 'gfw-google-827.alerts.satellite_timing_offsets.3-0-0',
 'gfw-google-827.alerts.segment_identity_daily_.2-5',
 'gfw-google-827.alerts.segment_identity_daily_.3-0-0',
 'gfw-google-827.a

In [12]:
TABLE_NAME = "encounters"
DUMMY_BATCH_DATE = '2023-04-01' # the date slice you want to run interactive expectations on

In [13]:
for current_expectation_suite_name in [es for es in context.list_expectation_suite_names() if TABLE_NAME in es and 'constraints' in es]:
    print(current_expectation_suite_name)
    current_expectation_suite=context.get_expectation_suite(current_expectation_suite_name)    
    current_expectation_suite_asset_name=current_expectation_suite.meta.get('asset_name')
    current_expectation_suite_datasource_name=current_expectation_suite.meta.get('datasource_name')
    current_expectation_suite_version_number=current_expectation_suite.meta.get('version_number')

    gx_asset=gx_datasource.get_asset(current_expectation_suite_asset_name)
    gx_splitter=gx_asset.splitter
    if gx_splitter is not None:
        DATE_PARTITION_COLUMN=gx_splitter.column_name
        br_options={DATE_PARTITION_COLUMN: DUMMY_BATCH_DATE}
    else:
        br_options={}
    gx_br = gx_asset.build_batch_request(br_options)
    gx_batches = gx_datasource.get_batch_list_from_batch_request(gx_br)
    gx_validator = context.get_validator_using_batch_list(current_expectation_suite, gx_batches)



    gx_validator.expect_column_values_to_be_unique('encounter_id')
    gx_validator.expect_column_values_to_not_be_null('encounter_id')


    gx_validator.expect_column_pair_values_a_to_be_greater_than_b('end_time', 'start_time')
    gx_validator.expect_queried_custom_query_to_return_num_rows(template_dict={"user_query": f"""
        SELECT *
        FROM {{active_batch}}
        WHERE DATETIME_DIFF(end_time, start_time, hour) < 2
    """}, value=0, meta={
                "notes": {
                    "format": "markdown",
                    "content": "Encounters have a length of at least 2 hours (https://globalfishingwatch.slack.com/archives/C4E5GR5AN/p1689200160767819).",
            }
    })

    gx_validator.save_expectation_suite(discard_failed_expectations=False)

DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
DEBUG:google.cloud.bigquery.opentelemetry_tracing:This service is instrumented using OpenTelemetry. OpenTelemetry or one of its components could not be imported; please add compatible versions of opentelemetry-api and opentelemetry-instrumentation packages in order to get BigQuery Tracing data.


DEBUG:urllib3.util.retry:Converted retries value: 3 -> Retry(total=3, connect=None, read=None, redirect=None, status=None)
DEBUG:google.auth.transport.requests:Making request: POST https://oauth2.googleapis.com/token
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): oauth2.googleapis.com:443


gfw-google-827.constraints.encounters.2-5


DEBUG:urllib3.connectionpool:https://oauth2.googleapis.com:443 "POST /token HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): bigquery.googleapis.com:443
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/7af7dc9c-31ab-4f71-9f29-201d36d7fb41?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.__fields_set__ assets added
INFO:great_expectations.datasource.fluent.fluent_base_model:SQLDatasource.dict() - substituting config values
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigque

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/datasets/tech_great_expectations_temp_ttl_7d/tables/gx_temp_fcf24536?prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:Attempting query SELECT count(*) AS "table.row_count" 
FROM gx_temp_fcf24536
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/f341516c-0ec6-4faf-8e60-4fdf64137ccf?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
DEBUG:great_expectations.execution_engine.sqlalchemy_execution_engine:SqlAlchemyExecutionEngine computed 1 metrics on domain_id 70a4d1881abdcf505ed07895cc917a2f
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "POST /bigquery/v2/projects/world-fishing-827/jobs?prettyPrint=false HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:https://bigquery.googleapis.com:443 "GET /bigquery/v2/projects/world-fishing-827/queries/28a1826e-992b-4fd4-9ff1-f993aaf81ce1?maxResults=0&location=US&prettyPrint=false HTTP/1.1" 200 None
INFO:great_expectations.validator.validator:	4 expectation(s) included in expectation_suite.
INFO:great_expectations.validator.validator:	4 expectation(s) included in expectation_suite.


DEBUG:urllib3.connectionpool:https://stats.greatexpectations.io:443 "POST /great_expectations/v1/usage_statistics HTTP/1.1" 201 18
DEBUG:great_expectations.core.usage_statistics.usage_statistics:Posted usage stats: message status 201
